In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_csv("Bawang Merah.csv")

In [3]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [4]:
df.head()

,Date,Aceh,Bali,Banten,Bengkulu,DI Yogyakarta,DKI Jakarta,Gorontalo,Jambi,Jawa Barat,...,Papua,Riau,Sulawesi Barat,Sulawesi Selatan,Sulawesi Tengah,Sulawesi Tenggara,Sulawesi Utara,Sumatera Barat,Sumatera Selatan,Sumatera Utara
0,2022-01-01,28970.0,20870.0,26890.0,26650.0,25240.0,35510.0,31740.0,23390.0,25800.0,...,48610.0,29240.0,25160.0,24910.0,28030.0,30380.0,35750.0,23360.0,26670.0,28710.0
1,2022-01-02,29900.0,20710.0,25600.0,26950.0,25240.0,31850.0,30020.0,23550.0,26010.0,...,50160.0,28750.0,24770.0,24360.0,27200.0,30260.0,35630.0,23790.0,25690.0,28460.0
2,2022-01-03,28970.0,20510.0,26390.0,27290.0,24620.0,34880.0,31250.0,23730.0,25910.0,...,49510.0,27870.0,24140.0,24740.0,26750.0,30080.0,34980.0,22620.0,26270.0,28050.0
3,2022-01-04,29600.0,20180.0,26630.0,27450.0,24370.0,35260.0,31640.0,23300.0,25950.0,...,49670.0,28330.0,24450.0,24710.0,28800.0,29950.0,34920.0,23010.0,26800.0,27800.0
4,2022-01-05,29540.0,19960.0,26610.0,27710.0,24210.0,35260.0,34010.0,23640.0,25700.0,...,46590.0,28240.0,24640.0,24780.0,28990.0,30240.0,34000.0,23330.0,25360.0,27670.0


In [5]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.000000
Aceh                         3.685259
Bali                         3.585657
Banten                       3.685259
Bengkulu                     3.685259
DI Yogyakarta                3.585657
DKI Jakarta                  3.685259
Gorontalo                    3.486056
Jambi                        3.784861
Jawa Barat                   3.685259
Jawa Tengah                  3.386454
Jawa Timur                   3.486056
Kalimantan Barat             3.585657
Kalimantan Selatan           3.685259
Kalimantan Tengah            3.585657
Kalimantan Timur             3.884462
Kalimantan Utara             3.884462
Kepulauan Bangka Belitung    3.784861
Kepulauan Riau               3.884462
Lampung                      3.685259
Maluku Utara                 3.585657
Maluku                       3.685259
Nusa Tenggara Barat          3.685259
Nusa Tenggara Timur          3.386454
Papua Barat                  3.884462
Papua                        3.685259
Riau        

In [6]:
numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())

In [7]:
zero_var_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with zero variance:", zero_var_cols)

Columns with zero variance: []


In [8]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.0
Aceh                         0.0
Bali                         0.0
Banten                       0.0
Bengkulu                     0.0
DI Yogyakarta                0.0
DKI Jakarta                  0.0
Gorontalo                    0.0
Jambi                        0.0
Jawa Barat                   0.0
Jawa Tengah                  0.0
Jawa Timur                   0.0
Kalimantan Barat             0.0
Kalimantan Selatan           0.0
Kalimantan Tengah            0.0
Kalimantan Timur             0.0
Kalimantan Utara             0.0
Kepulauan Bangka Belitung    0.0
Kepulauan Riau               0.0
Lampung                      0.0
Maluku Utara                 0.0
Maluku                       0.0
Nusa Tenggara Barat          0.0
Nusa Tenggara Timur          0.0
Papua Barat                  0.0
Papua                        0.0
Riau                         0.0
Sulawesi Barat               0.0
Sulawesi Selatan             0.0
Sulawesi Tengah              0.0
Sulawesi T

In [9]:
numerical_features = df.select_dtypes(include=['number'])
categorical_features = df.select_dtypes(exclude=['number'])

# Apply VarianceThreshold to remove low-variance numerical features
selector = VarianceThreshold(threshold=0.01)  # Adjust threshold as needed
reduced_numerical_df = selector.fit_transform(numerical_features)

# Convert back to DataFrame with selected features
reduced_numerical_df = pd.DataFrame(reduced_numerical_df, 
                                       columns=numerical_features.columns[selector.get_support()])

# Combine numerical and categorical features back together
reduced_df = pd.concat([reduced_numerical_df, categorical_features.reset_index(drop=True)], axis=1)

In [10]:
df.shape

(1004, 35)

In [11]:
reduced_df.shape

(1004, 35)

In [12]:
skewness = df.select_dtypes(include=['number']).apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
print(skewness)

Kalimantan Utara             1.851168
Kalimantan Timur             1.797828
Sulawesi Barat               1.596922
Kalimantan Barat             1.502862
Sulawesi Tengah              1.435748
Aceh                         1.381457
Maluku Utara                 1.374832
Banten                       1.355453
Jawa Barat                   1.311847
Kalimantan Selatan           1.304325
Sulawesi Utara               1.276241
DKI Jakarta                  1.263061
Sulawesi Selatan             1.248429
Kalimantan Tengah            1.165747
Sumatera Utara               1.157323
Sulawesi Tenggara            1.136730
Papua Barat                  1.097514
Papua                        1.096101
Maluku                       1.048781
Sumatera Selatan             1.027960
Kepulauan Bangka Belitung    1.004639
Gorontalo                    0.962599
Lampung                      0.962473
Jawa Tengah                  0.958010
Jawa Timur                   0.911671
Bali                         0.859513
Bengkulu    

In [13]:
# Select numerical columns
num_cols = df.select_dtypes(include=['number'])

# Function to calculate outlier percentage using IQR
def outlier_percentage(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((column < lower_bound) | (column > upper_bound)).sum()
    return (outliers / len(column)) * 100  # Percentage

# Apply function to all numerical columns
outlier_percentages_df = num_cols.apply(outlier_percentage)

# Display the results
print(outlier_percentages_df.sort_values(ascending=False))

Sumatera Utara               13.047809
Bengkulu                     13.047809
Sumatera Barat               10.756972
Aceh                          9.960159
Jambi                         7.569721
Riau                          7.270916
Kalimantan Utara              6.872510
Sumatera Selatan              6.772908
DKI Jakarta                   6.772908
Banten                        5.976096
Bali                          5.577689
Sulawesi Utara                4.980080
Sulawesi Tengah               4.382470
Maluku Utara                  4.282869
Papua Barat                   3.884462
Jawa Barat                    3.784861
Jawa Timur                    3.685259
Sulawesi Tenggara             3.386454
Kepulauan Bangka Belitung     3.286853
Papua                         3.187251
DI Yogyakarta                 3.187251
Jawa Tengah                   3.087649
Maluku                        2.988048
Kalimantan Timur              2.988048
Sulawesi Barat                2.988048
Lampung                  

In [14]:
test = pd.read_csv("Bawang Merah Test.csv")

In [15]:
df.to_csv("Bawang Merah Clean.csv", index=False)

In [16]:
def df_to_X_y(df, window_size=5):
  df_as_np = df.to_numpy()
  X = []
  y = []
  for i in range(len(df_as_np)-window_size):
    row = [[a] for a in df_as_np[i:i+window_size]]
    X.append(row)
    label = df_as_np[i+window_size]
    y.append(label)
  return np.array(X), np.array(y)

In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = df.select_dtypes(include=[np.number])
df = df.apply(pd.to_numeric, errors='coerce').dropna()

if df.shape[1] > 1:
    print(f"Warning: DataFrame has multiple numeric columns ({df.shape[1]}). Using the first column.")
    df = df.iloc[:, 0]

# StandardScaler
scaler = StandardScaler()
# scaler = MinMaxScaler(feature_range=(0, 1))
df = scaler.fit_transform(df.values.reshape(-1, 1))

def df_to_X_y(df, window_size=5):
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size])
        y.append(df[i + window_size])
    return np.array(X), np.array(y)

window_size = 5
X, y = df_to_X_y(df, window_size)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

checkpoint_path = "model_checkpoint.keras"
cp4 = ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,  # best model
    monitor='val_loss',
    mode='min'
)

lr_reducer = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.8, 
    patience=5, 
    min_lr=1e-6, 
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,  
    restore_best_weights=True,
    verbose=1
)

def create_lstm_model(input_shape):
    model = Sequential([
        InputLayer(input_shape=input_shape),
       # LSTM(256, return_sequences=True),
       # Dropout(0.3),  # Increase dropout
        LSTM(128, return_sequences=True),
        Dropout(0.3),  # Increase dropout
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])
    model.compile(loss='mape', optimizer=Adam(learning_rate=0.001), metrics=['mape'])
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
model = create_lstm_model(input_shape)
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), 
          epochs=100, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Training completed. Final epoch:", len(history.history['loss']))

model.save("lstm_model.keras")

# sample submission
df_submission = pd.read_csv("sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)
print("Total required predictions:", total_required_predictions)
print("Unique countries:", len(unique_countries))

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, window_size, 1))[0, 0]
    pred += np.random.normal(0, 0.01)  
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

# lebih dari 3128
if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

# converting the predicted values back to their original scale
y_pred = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

# submission
data = []
for i in range(min(len(df_submission), len(y_pred))):
    data.append({'id': df_submission.iloc[i]['id'], 'price': y_pred[i][0]})

submission_df = pd.DataFrame(data)
submission_df.to_csv("bawang_merah_submission.csv", index=False)
print("Submission file saved as bawang_merah_submission.csv")

c:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning:

Argument `input_shape` is deprecated. Use `shape` instead.



Epoch 1/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - loss: 100.2864 - mape: 100.2864 - val_loss: 91.3169 - val_mape: 91.3169 - learning_rate: 0.0010
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 86.4583 - mape: 86.4583 - val_loss: 77.2793 - val_mape: 77.2793 - learning_rate: 0.0010
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 65.1571 - mape: 65.1571 - val_loss: 77.5375 - val_mape: 77.5375 - learning_rate: 0.0010
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 62.9373 - mape: 62.9373 - val_loss: 75.0515 - val_mape: 75.0515 - learning_rate: 0.0010
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 67.2635 - mape: 67.2635 - val_loss: 79.7603 - val_mape: 79.7603 - learning_rate: 0.0010
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 63.3732 - mape: 63.3732 - val_loss: 72.8240 - val_mape: 72.8240 - learning_rate: 0.0010
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 65.7919 - mape: 65.7919 - val_loss: 65.6867 - val_